In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
import platform
import matplotlib.pyplot as plt

# 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [7]:
# ============================================================
# 1. 전처리된 데이터 로드
# ============================================================
df_portfolio = pd.read_csv("data/portfolio.csv", index_col=0)
df_profile = pd.read_csv("clean_data/profile_cleaned.csv", index_col=0)
df_transcript = pd.read_csv("clean_data/transcript_cleaned.csv", index_col=0)

In [8]:
# ============================================================
# 2. 데이터 기본 구조 확인
# ============================================================
print("===== 기본 크기 확인 =====")
print("\n[portfolio]")
print("shape :", df_portfolio.shape)
display(df_portfolio.head())
print()

print("[profile]")
print("shape :", df_profile.shape)
display(df_profile.head())
print()

print("[transcript]")
print("shape :", df_transcript.shape)
display(df_transcript.head())
print()

===== 기본 크기 확인 =====

[portfolio]
shape : (10, 6)


,reward,channels,difficulty,duration,offer_type,id
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7



[profile]
shape : (14825, 5)


,gender,age,id,became_member_on,income
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0
5,M,68,e2127556f4f64592b11af22de27a7932,20180426,70000.0
8,M,65,389bc3fa690240e798340f5a15918d5c,20180209,53000.0
12,M,58,2eeac8d8feae4a8cad5a6af0499a211d,20171111,51000.0



[transcript]
shape : (306137, 7)


,person,event,value,time,offer_id,amount,reward_value
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN


In [9]:
# ============================================================
# 2. 각 데이터셋 결측치 최종 확인
# ============================================================
print("="*60)
print("[portfolio] 결측치 확인")
print(df_portfolio.isna().sum())
print("="*60)

print("="*60)
print("[profile] 결측치 확인")
print(df_profile.isna().sum())
print("="*60)

print("="*60)
print("[transcript] 결측치 확인")
print(df_transcript.isna().sum())
print("="*60)

[portfolio] 결측치 확인
reward        0
channels      0
difficulty    0
duration      0
offer_type    0
id            0
dtype: int64
[profile] 결측치 확인
gender              0
age                 0
id                  0
became_member_on    0
income              0
dtype: int64
[transcript] 결측치 확인
person               0
event                0
value                0
time                 0
offer_id        138953
amount          167184
reward_value    272955
dtype: int64


컬럼	            값이 있는 이벤트	              결측이 많은 이유
offer_id	    received, viewed, completed	    transaction에는 없음
amount	            transaction	                 오퍼 이벤트에는 없음
reward_value	    completed	                나머지 이벤트에는 없음

In [10]:
print(df_transcript.groupby("event")[["offer_id", "amount", "reward_value"]].count())

                 offer_id  amount  reward_value
event                                          
offer completed     33182       0         33182
offer received      76277       0             0
offer viewed        57725       0             0
transaction             0  138953             0


In [5]:
# 결측치 개수 + 비율 확인 함수
def check_na(df, name):
    na_count = df.isna().sum()
    na_ratio = (na_count / len(df) * 100).round(2)
    na_summary = pd.DataFrame({
        "결측치 개수": na_count,
        "결측치 비율(%)": na_ratio
    })
    
    print("="*60)
    print(f"[{name}] 결측치 요약")
    display(na_summary[na_summary["결측치 개수"] > 0])
    print("="*60)

check_na(df_portfolio, "portfolio")
check_na(df_profile, "profile")
check_na(df_transcript, "transcript")

[portfolio] 결측치 요약


,결측치 개수,결측치 비율(%)


[profile] 결측치 요약


,결측치 개수,결측치 비율(%)


[transcript] 결측치 요약


,결측치 개수,결측치 비율(%)
offer_id,306137,100.0
amount,306137,100.0
reward_value,306137,100.0


In [6]:
print(df_transcript["value"].head())
print(type(df_transcript["value"].iloc[0]))

0    {'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}
1    {'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}
2    {'offer id': '2906b810c7d4411798c6938adc9daaa5'}
3    {'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'}
4    {'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}
Name: value, dtype: str
<class 'str'>


In [ ]:
# ============================================================
# 3. merge를 위한 컬럼 정리
# ============================================================
df_profile = df_profile.reset_index().rename(columns={"id": "customer_id"})
df_portfolio = df_portfolio.reset_index().rename(columns={"id": "offer_id"})
df_transcript = df_transcript.rename(columns={"person": "customer_id"})

print(df_profile.columns)
print(df_portfolio.columns)
print(df_transcript.columns)